# MRI

In [16]:
import os
import pathlib
import cv2 as cv
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras import datasets
from tensorflow.keras import optimizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import load_img, img_to_array
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator as imgen
from tensorflow.keras.layers import Dense, Input, Conv2D, MaxPooling2D,Flatten , Dropout,BatchNormalization , GlobalAveragePooling2D

In [26]:
data_dir = 'dataset/mri'
img_size = (150,150)
batch_size = 32

In [23]:
# 학습용 데이터 : 데이터 증강 , 정규화
train_data = imgen(
    rescale=1/255, 
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
                  )

# 검증용 데이터 : 정규화
valid_data = imgen(rescale=1/255,validation_split=0.2)

In [24]:
train_gen = train_data.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    seed=2020
)

test_gen = valid_data.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    seed=2020
)

Found 226 images belonging to 3 classes.
Found 56 images belonging to 3 classes.


In [8]:
model_01 = Sequential()

model_01.add(Input(shape=(150,150,3)))

model_01.add(Conv2D(32,(3,3),activation='relu'))
model_01.add(BatchNormalization())
model_01.add(MaxPooling2D(2,2))

model_01.add(Conv2D(64,(3,3),activation='relu'))
model_01.add(BatchNormalization())
model_01.add(MaxPooling2D(2,2))

model_01.add(Conv2D(128,(3,3),activation='relu'))
model_01.add(BatchNormalization())
model_01.add(MaxPooling2D(2,2))

model_01.add(GlobalAveragePooling2D())
model_01.add(Dense(64,activation='relu'))
model_01.add(Dropout(0.5))
model_01.add(Dense(1,activation='sigmoid'))

model_01.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 148, 148, 32)      896       
                                                                 
 batch_normalization_3 (Batc  (None, 148, 148, 32)     128       
 hNormalization)                                                 
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 74, 74, 32)       0         
 2D)                                                             
                                                                 
 conv2d_4 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 batch_normalization_4 (Batc  (None, 72, 72, 64)       256       
 hNormalization)                                                 
                                                      

In [10]:
model_01.compile(optimizer=optimizers.Adam(learning_rate=0.0001), 
                 loss='binary_crossentropy', metrics=['accuracy'])

callback_01 = [EarlyStopping(monitor='val_accuracy', patience=10,
                            restore_best_weights=True, verbose=1)]

history_01 = model_01.fit(train_gen, validation_data = test_gen, 
                   epochs=100 , callbacks = callback_01)

Epoch 1/50
8/8 [==============================] - 2s 194ms/step - loss: -0.9884 - accuracy: 0.4248 - val_loss: 0.6159 - val_accuracy: 0.4286
Epoch 2/50
8/8 [==============================] - 1s 144ms/step - loss: -1.3716 - accuracy: 0.4248 - val_loss: 0.5382 - val_accuracy: 0.4286
Epoch 3/50
8/8 [==============================] - 1s 131ms/step - loss: -1.6870 - accuracy: 0.4248 - val_loss: 0.4497 - val_accuracy: 0.4286
Epoch 4/50
8/8 [==============================] - 1s 131ms/step - loss: -1.9182 - accuracy: 0.4248 - val_loss: 0.3505 - val_accuracy: 0.4286
Epoch 5/50
8/8 [==============================] - 1s 134ms/step - loss: -2.1979 - accuracy: 0.4248 - val_loss: 0.2451 - val_accuracy: 0.4286
Epoch 6/50
8/8 [==============================] - 1s 134ms/step - loss: -2.5025 - accuracy: 0.4248 - val_loss: 0.1072 - val_accuracy: 0.4286
Epoch 7/50
8/8 [==============================] - 1s 129ms/step - loss: -2.7458 - accuracy: 0.4248 - val_loss: -0.0296 - val_accuracy: 0.4286
Epoch 8/50
8

In [14]:
test_gen.reset()
pred = model_01.predict(test_gen)

pred_label = (pred>0.3).astype(int).reshape(-1)
true_label = test_gen.classes

print(confusion_matrix(true_label,pred_label))
print(classification_report(true_label,pred_label))

2/2 [==============================] - 0s 127ms/step
[[24  0]
 [32  0]]
              precision    recall  f1-score   support

           1       0.43      1.00      0.60        24
           2       0.00      0.00      0.00        32

    accuracy                           0.43        56
   macro avg       0.21      0.50      0.30        56
weighted avg       0.18      0.43      0.26        56



C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 전이 학습

In [28]:
# 학습용 데이터 : 데이터 증강 , 정규화
train_data = imgen(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

# 검증용 데이터 : 정규화
valid_data = imgen(
    rescale=1./255,
    validation_split=0.2
)

In [29]:
train_gen = train_data.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    seed=2020
)

test_gen = valid_data.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    seed=2020
)

Found 226 images belonging to 3 classes.
Found 56 images belonging to 3 classes.


In [36]:
base_model = VGG16(
    input_shape=(150,150,3),
    include_top = False,
    weights = 'imagenet'
)

base_model.trainable=True

for layer in base_model.layers[:30]:
    layer.trainable = False

model_02 = Sequential()

model_02.add(base_model)

model_02.add(GlobalAveragePooling2D())
model_02.add(Dense(64,activation='relu'))
model_02.add(Dropout(0.5))
model_02.add(Dense(1,activation='sigmoid'))

model_02.summary()

Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 4, 4, 512)         14714688  
                                                                 
 global_average_pooling2d_5   (None, 512)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_10 (Dense)            (None, 64)                32832     
                                                                 
 dropout_5 (Dropout)         (None, 64)                0         
                                                                 
 dense_11 (Dense)            (None, 1)                 65        
                                                                 
Total params: 14,747,585
Trainable params: 32,897
Non-trainable params: 14,714,688
_____________________________________

In [37]:
model_02.compile(optimizer=optimizers.Adam(learning_rate=0.000001), 
                 loss='binary_crossentropy', metrics=['accuracy'])

callback_02 = [EarlyStopping(monitor='val_loss', patience=10,
                            restore_best_weights=True, verbose=1)]

history_02 = model_02.fit(train_gen, validation_data = test_gen, 
                   epochs=100 , callbacks = callback_02)

Epoch 1/100
8/8 [==============================] - 2s 148ms/step - loss: 1.7820 - accuracy: 0.0354 - val_loss: 1.7825 - val_accuracy: 0.0000e+00
Epoch 2/100
8/8 [==============================] - 1s 126ms/step - loss: 1.7758 - accuracy: 0.0531 - val_loss: 1.7787 - val_accuracy: 0.0000e+00
Epoch 3/100
8/8 [==============================] - 1s 129ms/step - loss: 1.9074 - accuracy: 0.0398 - val_loss: 1.7748 - val_accuracy: 0.0000e+00
Epoch 4/100
8/8 [==============================] - 1s 142ms/step - loss: 1.8183 - accuracy: 0.0619 - val_loss: 1.7711 - val_accuracy: 0.0000e+00
Epoch 5/100
8/8 [==============================] - 1s 132ms/step - loss: 1.6789 - accuracy: 0.0442 - val_loss: 1.7677 - val_accuracy: 0.0000e+00
Epoch 6/100
8/8 [==============================] - 1s 138ms/step - loss: 1.7289 - accuracy: 0.0531 - val_loss: 1.7642 - val_accuracy: 0.0000e+00
Epoch 7/100
8/8 [==============================] - 1s 133ms/step - loss: 1.7691 - accuracy: 0.0575 - val_loss: 1.7607 - val_accura

In [38]:
test_gen.reset()
pred = model_02.predict(test_gen)

pred_label = (pred>0.5).astype(int).reshape(-1)
true_label = test_gen.classes

print(confusion_matrix(true_label,pred_label))
print(classification_report(true_label,pred_label))

2/2 [==============================] - 0s 38ms/step
[[ 0  0  0]
 [24  0  0]
 [32  0  0]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00      24.0
           2       0.00      0.00      0.00      32.0

    accuracy                           0.00      56.0
   macro avg       0.00      0.00      0.00      56.0
weighted avg       0.00      0.00      0.00      56.0



C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\pc\anaconda3\envs\dl_gpu\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\pc\anaconda3